# 02 — Build the three training sets
All three fine-tuning runs use the **same 100K-or-fewer sample budget rule** so the comparison isolates one thing:

| Tag | Data | Samples seen |
|---|---|---|
| `raw1x` | raw train images, each once | ~train size |
| `rawrepeat` | same raw images cycled (reshuffled each cycle) | 100,000 |
| `aug100k` | random 100K from the augmented pool | 100,000 |

`rawrepeat` vs `aug100k` has equal compute and the same source images. The only difference is augmentation.

In [ ]:
import numpy as np, pandas as pd
import config as C

tr = pd.read_csv(C.SPLIT_DIR / "train.csv")
va = pd.read_csv(C.SPLIT_DIR / "val.csv")
aug = pd.read_csv(C.AUG_FINAL / "augmented.csv")
print(f"raw train: {len(tr):,} | augmented pool: {len(aug):,} | val: {len(va):,}")

def raw_path(fn): return str(C.RAW_IMG_DIR / fn)
def aug_path(fn): return str(C.AUG_FINAL / "images" / fn)

def frame(d, pathfn): return pd.DataFrame({"image": d["file_name"].map(pathfn), "text": d["text"]})

# raw1x
raw1x = frame(tr, raw_path).sample(frac=1, random_state=C.SEED).reset_index(drop=True)

# rawrepeat: cycle the raw images until N_TRAIN_SAMPLES, reshuffling each cycle
rng = np.random.default_rng(C.SEED)
base = frame(tr, raw_path)
parts, total = [], 0
while total < C.N_TRAIN_SAMPLES:
    parts.append(base.iloc[rng.permutation(len(base))]); total += len(base)
rawrepeat = pd.concat(parts, ignore_index=True).iloc[:C.N_TRAIN_SAMPLES]

# aug100k
aug100k = frame(aug.sample(n=C.N_TRAIN_SAMPLES, random_state=C.SEED), aug_path).reset_index(drop=True)

# small validation set used only to track validation loss
val_loss = frame(va.sample(n=min(C.N_VAL_LOSS, len(va)), random_state=C.SEED), raw_path).reset_index(drop=True)

for name, d in (("raw1x", raw1x), ("rawrepeat", rawrepeat), ("aug100k", aug100k), ("val_loss", val_loss)):
    d.to_csv(C.SUBSET_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name:10s} {len(d):>8,} samples  ->  {len(d)//8:,} optimizer steps (effective batch 8)" if name != "val_loss" else f"{name:10s} {len(d):>8,} samples")

In [ ]:
# integrity: every image file must exist, and no training image may be a val/test image
import json
sp = json.load(open(C.SPLIT_DIR / "splits.json"))
held = set(pd.concat([pd.read_csv(C.SPLIT_DIR / "val.csv"), pd.read_csv(C.SPLIT_DIR / "test.csv")])["file_name"])
for name in ("raw1x", "rawrepeat", "aug100k"):
    d = pd.read_csv(C.SUBSET_DIR / f"{name}.csv")
    import os
    miss = sum(not os.path.exists(p) for p in d["image"].sample(min(3000, len(d)), random_state=0))
    leak = sum(os.path.basename(p) in held for p in d["image"])
    print(f"{name:10s} missing files (sampled): {miss} | val/test files inside: {leak}")
    assert miss == 0 and leak == 0